# Milestone 3

Saebastion Cole

10-31-26

## Pull Data from an API: Wikipedia Intentional Homicide Rate

In [22]:
# Imports
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
# Make the API call to the Wikipedia page
# Wikipedia rejects requests without a User-Agent, so I set one.
url = "https://en.wikipedia.org/wiki/List_of_countries_by_intentional_homicide_rate"
headers = {"User-Agent": "data-preparation-hw/1.0 (student project)"}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
# A 200 is successful
print(f"Status code: {response.status_code}")

Status code: 200


In [24]:
# Parse the HTML and find the homicide rate table
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table", class_="wikitable")

# The first row holds the column headers, the rest are the data rows
rows = table.find_all("tr")
columns = [th.get_text(strip=True) for th in rows[0].find_all("th")]
data = [[td.get_text(strip=True) for td in tr.find_all("td")] for tr in rows[1:]]

In [25]:
# Load the table into a single data frame
homicide_df = pd.DataFrame(data, columns=columns)
print(homicide_df.shape)
homicide_df.head()

(205, 5)


,Location,Rate,Year,Region,Subregion
0,Afghanistan *,4.224,2023,Asia,Southern Asia
1,Albania *,1.755,2024,Europe,Southern Europe
2,Algeria,1.382,2024,Africa,Northern Africa
3,American Samoa,0,2019,Oceania,Polynesia
4,Andorra,2.585,2020,Europe,Southern Europe


## Transformation 1: convert the year to a date format
The Year column came in as text. I will convert it to an actual datetime so it can be sorted, filtered, and compared as a date. Since only the year is given, each value becomes January 1st of that year.

In [26]:
# First data manipulation: convert Year from text to a datetime
print(f"Year type before: {homicide_df['Year'].dtype}")

# errors='coerce' turns anything that isn't a valid year into NaT instead of crashing
homicide_df['Year'] = pd.to_datetime(homicide_df['Year'], format='%Y', errors='coerce')

print(f"Year type after: {homicide_df['Year'].dtype}")
print(f"Years that failed to convert: {homicide_df['Year'].isna().sum()}")
homicide_df.head()

Year type before: str
Year type after: datetime64[us]
Years that failed to convert: 0


,Location,Rate,Year,Region,Subregion
0,Afghanistan *,4.224,2023-01-01,Asia,Southern Asia
1,Albania *,1.755,2024-01-01,Europe,Southern Europe
2,Algeria,1.382,2024-01-01,Africa,Northern Africa
3,American Samoa,0,2019-01-01,Oceania,Polynesia
4,Andorra,2.585,2020-01-01,Europe,Southern Europe


## Transformation 2: format the rate into a more readable format
The Rate column is text with up to three decimals. I will convert it to a number so it can be used in calculations, and round it to two decimals so it is easier to read.

In [27]:
# Second data manipulation: convert Rate from text to a number and round it
print(f"Rate type before: {homicide_df['Rate'].dtype}")

homicide_df['Rate'] = pd.to_numeric(homicide_df['Rate'], errors='coerce').round(2)

print(f"Rate type after: {homicide_df['Rate'].dtype}")
print(f"Rates that failed to convert: {homicide_df['Rate'].isna().sum()}")
homicide_df.head()

Rate type before: str
Rate type after: float64
Rates that failed to convert: 0


,Location,Rate,Year,Region,Subregion
0,Afghanistan *,4.22,2023-01-01,Asia,Southern Asia
1,Albania *,1.76,2024-01-01,Europe,Southern Europe
2,Algeria,1.38,2024-01-01,Africa,Northern Africa
3,American Samoa,0.00,2019-01-01,Oceania,Polynesia
4,Andorra,2.58,2020-01-01,Europe,Southern Europe


## Transformation 3: identify outliers and bad data
I will flag three things instead of deleting them, so they can be filtered later:
- **Outliers:** rates above the upper IQR fence (Q3 + 1.5 * IQR).
- **Zero rates:** a rate of 0 usually means a very small territory or missing reporting, not truly zero homicides.
- **Old data:** figures from before 2010 are too stale to compare fairly against recent years.

In [28]:
# Third data manipulation: flag outliers, zero rates, and old data
q1 = homicide_df['Rate'].quantile(0.25)
q3 = homicide_df['Rate'].quantile(0.75)
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"Upper outlier fence: {upper_fence:.2f}")

homicide_df['Rate Outlier'] = homicide_df['Rate'] > upper_fence
homicide_df['Zero Rate'] = homicide_df['Rate'] == 0
homicide_df['Old Data'] = homicide_df['Year'] < pd.Timestamp('2010-01-01')

print(f"Outliers: {homicide_df['Rate Outlier'].sum()}")
print(f"Zero rates: {homicide_df['Zero Rate'].sum()}")
print(f"Old data: {homicide_df['Old Data'].sum()}")
homicide_df[homicide_df['Rate Outlier']].sort_values('Rate', ascending=False).head(10)

Upper outlier fence: 16.11
Outliers: 24
Zero rates: 10
Old data: 12


,Location,Rate,Year,Region,Subregion,Rate Outlier,Zero Rate,Old Data
189,Turks and Caicos Islands,103.15,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
74,Haiti *,64.34,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
154,Saint Kitts and Nevis *,59.77,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
158,Saint Vincent and the Grenadines,50.69,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
196,U.S. Virgin Islands,49.86,2012-01-01,Americas,Latin America and the Caribbean,True,False,False
185,Trinidad and Tobago *,41.52,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
155,Saint Lucia *,40.61,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
90,Jamaica *,40.36,2024-01-01,Americas,Latin America and the Caribbean,True,False,False
171,South Africa *,39.72,2024-01-01,Africa,Sub-Saharan Africa,True,False,False
50,Ecuador *,38.94,2024-01-01,Americas,Latin America and the Caribbean,True,False,False


## Transformation 4: fix inconsistent values
Some location names end with a " \*" marker (for example "Afghanistan \*"), while others do not. On Wikipedia this marker only means the country has a "Crime in ..." article, so it is not part of the name. I will remove it so names are consistent and can be matched against other data sets.

In [29]:
# Fourth data manipulation: remove the * marker and the narrow space before it from location names
print(f"Names with marker before: {homicide_df['Location'].str.contains('*', regex=False).sum()}")

homicide_df['Location'] = homicide_df['Location'].str.replace('*', '', regex=False).str.strip()

print(f"Names with marker after: {homicide_df['Location'].str.contains('*', regex=False).sum()}")
homicide_df.head()

Names with marker before: 123
Names with marker after: 0


,Location,Rate,Year,Region,Subregion,Rate Outlier,Zero Rate,Old Data
0,Afghanistan,4.22,2023-01-01,Asia,Southern Asia,False,False,False
1,Albania,1.76,2024-01-01,Europe,Southern Europe,False,False,False
2,Algeria,1.38,2024-01-01,Africa,Northern Africa,False,False,False
3,American Samoa,0.00,2019-01-01,Oceania,Polynesia,False,True,False
4,Andorra,2.58,2020-01-01,Europe,Southern Europe,False,False,False


## Transformation 5: replace the headers
I will rename the columns to kebab casing to match the format I used in Milestone 2. I am also renaming Rate to make it clear it is homicides per 100,000 people.

In [30]:
# Fifth data manipulation: rename Rate to be descriptive, then convert all headers to kebab casing
homicide_df = homicide_df.rename(columns={'Rate': 'Homicide Rate Per 100k'})
homicide_df.columns = homicide_df.columns.str.lower().str.replace(' ', '-')

homicide_df.head()

,location,homicide-rate-per-100k,year,region,subregion,rate-outlier,zero-rate,old-data
0,Afghanistan,4.22,2023-01-01,Asia,Southern Asia,False,False,False
1,Albania,1.76,2024-01-01,Europe,Southern Europe,False,False,False
2,Algeria,1.38,2024-01-01,Africa,Northern Africa,False,False,False
3,American Samoa,0.00,2019-01-01,Oceania,Polynesia,False,True,False
4,Andorra,2.58,2020-01-01,Europe,Southern Europe,False,False,False


In [ ]:
# Lastly write the dataset to a CSV file
homicide_df.to_csv('data/homicide-rates.csv', index=False)

## Ethical Implications

The ethical implications of these changes are quite minor. The data itself has not been modified but instead the presentation has been made cleaner. Updating things like the headers will make the aggregation more normal when I combine the other data sources. The Wikipedia source is credible as this is a very standard maintained page (some of the other ones allow random contribution). We can assume the data is acquired ethically from each country's own dedicated researching. This though may leave out some homicides that there are not good record keeping to track.